# Lesson 12 — Honest uncertainty

**You will:** make an agent flag what it doesn't know, and use that data downstream.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/12-honest-uncertainty/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

LLMs are trained to sound confident. They will state the wrong year for a historical event in the same tone they state the right one. For a chat toy, that's annoying. For an agent making decisions, it's a hazard.

The fix isn't 'make the model more accurate' — that's an entire research field. The fix is to make the agent **say what it doesn't know**. BareBear treats uncertainty as data: *assumptions* the agent had to make, and *missing information* it couldn't find. Both are surfaced in the `Report` as plain lists you can read, log, or branch on.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")

# Optional: pin a specific free-tier model. If unset, barebear uses its default.
# Free-tier model availability rotates — swap here if a lesson cell errors with
# "model not available". Any *:free model on https://openrouter.ai/models works.
# os.environ["BAREBEAR_MODEL"] = "meta-llama/llama-3.2-3b-instruct:free"

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")
print("Model:", os.environ.get("BAREBEAR_MODEL", "(framework default)"))

## A research agent that has to admit defeat

In [ ]:
from barebear import Bear, Task, Tool, OpenRouterModel

def search_db(query: str) -> str:
    return "No matching records."   # the trap — agent must admit it can't find anything

system_prompt = (
    "You are a research assistant. ALWAYS state your assumptions explicitly. "
    "If you cannot verify something, say 'I could not confirm X' clearly. "
    "It is better to say you don't know than to guess confidently."
)

bear = Bear(
    model=OpenRouterModel(),
    tools=[Tool(name="search_db", fn=search_db, description="Search internal records")],
)

task = Task(
    goal="Find the lifetime value of customer Acme Corp.",
    system_prompt=system_prompt,
)

report = bear.run(task, trace=True)
print()
print("--- final answer ---")
print(report.final_output)
print()
print("--- assumptions ---")
for a in report.assumptions:
    print(" •", a)
print()
print("--- uncertainties ---")
for u in report.uncertainties:
    print(" •", u)

## What to do with uncertainty data

Production code branches on these lists. For example:

```python
if report.uncertainties:
    escalate_to_human(report)
else:
    auto_approve(report)
```

The agent is no longer a black box that 'sounds confident' — it's a system that tells you when it's guessing.

## Exercise

1. Read the `_extract_uncertainty` method in [`src/barebear/bear.py`](https://github.com/richey-malhotra/barebear/blob/main/src/barebear/bear.py). It uses keyword matching. What are its weaknesses?
2. Compare the assumptions extracted *with* and *without* the explicit system prompt above. Does prompting for honesty produce richer uncertainty data?

## You're done

You've built every core agentic pattern from first principles:

| #  | Concept               | Built                |
|----|-----------------------|----------------------|
| 1  | What an LLM is        | direct call          |
| 2  | The agent loop        | one-tool agent       |
| 3  | Tool design           | three-tool agent     |
| 4  | State and memory      | note-taker           |
| 5  | Budgets               | runaway-loop trap    |
| 6  | Policy                | blocked tool         |
| 7  | Checkpoints           | email-approval flow  |
| 8  | Planning              | code-refactor preview|
| 9  | Reflection            | critique-and-revise  |
| 10 | Multi-agent           | research-then-write  |
| 11 | Evaluation            | agent test suite     |
| 12 | Honest uncertainty    | flagged assumptions  |

The framework is open. Read it. Modify it. Break it. Submit a lesson.

[Back to the syllabus](../README.md)

## You're done

You've worked through all 12 lessons. The framework is now an open book — read it, modify it, build with it. If you teach with this, [let us know](https://github.com/richey-malhotra/barebear/issues).